***Integrantes:** Jhoan Avila Gutierrez;*<br>
***Clase:** Procesamiento Lenguaje Natural*<br>
***Programa:** Esp. Inteligencia Artificial, Universidad de Cundinamarca*

# 🚀 Laboratorio de Procesamiento de Texto (NLP)

Este notebook contiene la implementación de los conceptos solicitados en la actividad para el procesamiento de texto:

**Primer Bloque:**
1. Normalización de Texto
2. Stemming
3. Stopwords
4. Term Frequency
5. Inverse Document Frequency
6. Part-of-Speech Tagging
7. Lematización
8. Parsing Sintáctico
9. Named-Entity

**Segundo Bloque:**
1. Generacion de texto usando LLM

**Tercer Bloque:**
1. Text Classification
2. Question Answering
3. Summarization
4. Sentence Similarity
5. Translation
6. Text Generation
7. Text mining

## 📦 0. Instalación y Configuración del Entorno
Ejecutar la siguiente celda para instalar y descargar los paquetes necesarios en el ambiente local (`NLTK`, `SpaCy`, `Scikit-Learn`, `Pandas`, etc.).

In [4]:
# Instalación de librerías en Google Colab
!pip install -q nltk spacy scikit-learn pandas gensim matplotlib

import nltk
import spacy
from spacy import displacy
import pandas as pd
import numpy as np
import re
import unicodedata
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# Descarga de recursos de NLTK
resources = ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger']
for r in resources:
    try:
        nltk.download(r, quiet=True)
    except Exception as e:
        print(f"Error descargando {r}: {e}")

# Descarga de modelo SpaCy en español
try:
    nlp_es = spacy.load("es_core_news_sm")
except OSError:
    !python -m spacy download es_core_news_sm
    nlp_es = spacy.load("es_core_news_sm")

print("✅ Entorno de NLP configurado exitosamente.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 34.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 88.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Entorno de NLP configurado exitosamente.


## ⚙️ Configuración Primer Bloque: `Class NLPProcesadorTexto`
1. Ingresamos el texto que sera procesado por cada funcion.
2. Encapsulamos todas las funciones de procesamiento de texto del primer bloque en una clase estructurada.

In [41]:
texto_a_procesar = """
El grave incendio forestal en Villa de Leyva (Boyacá), iniciado el pasado sábado diecinueve de septiembre
en la vereda La Castellana y el cerro San Marcos, ha consumido más de 1340 hectáreas de vegetación nativa y pino,
afectando gravemente zonas de acuíferos, senderos turísticos y forzando la suspensión temporal de clases.
Las autoridades, que reportan un control del fuego cercano al 78%, sospechan que la emergencia fue provocada
por manos criminales, por lo que la Gobernación de Boyacá ha ofrecido una recompensa de hasta cuarenta y cinco millones
de pesos por información de los responsables mientras los cuerpos de socorro y helicópteros continúan extinguiendo
los focos activos.
"""

class NLPProcesadorTexto:
    def __init__(self, lang="es"):
        self.lang = lang
        nltk_lang = 'spanish' if self.lang == 'es' else 'english'
        self.stemmer = nltk.stem.SnowballStemmer(nltk_lang)
        self.nltk_stopwords = set(nltk.corpus.stopwords.words(nltk_lang))
        model_name = "es_core_news_sm" if self.lang == "es" else "en_core_web_sm"
        self.nlp = spacy.load(model_name)

    # 1. Normalización
    def normalizar_texto(self, texto, remover_acentos=True, remover_numeros=False, minusculas=True):
        if minusculas:
            texto = texto.lower()
        if remover_acentos:
            texto = unicodedata.normalize('NFD', texto)
            texto = ''.join([c for c in texto if unicodedata.category(c) != 'Mn'])
        if remover_numeros:
            texto = re.sub(r'\d+', '', texto)
        texto = re.sub(r'[^\w\s]', '', texto)
        return re.sub(r'\s+', ' ', texto).strip()

    # 2. Stemming
    def stemming_texto(self, tokens):
        if isinstance(tokens, str):
            tokens = re.findall(r'\b\w+\b', tokens)
        return [(t, self.stemmer.stem(t)) for t in tokens]

    # 3. Stopwords
    def remover_stopwords(self, tokens, adicionales=None):
        stops = set(self.nltk_stopwords).union(self.nlp.Defaults.stop_words)
        if adicionales:
            stops.update([a.lower() for a in adicionales])
        return [t for t in tokens if t.lower() not in stops and len(t) > 1]

    # 4. Term Frequency (TF)
    def calcular_tf(self, corpus):
        vectorizer = CountVectorizer()
        matriz = vectorizer.fit_transform(corpus)
        return pd.DataFrame(matriz.toarray(), columns=vectorizer.get_feature_names_out(), index=[f'Texto_{i+1}' for i in range(len(corpus))])

    # 5. TF-IDF & IDF
    def calcular_tfidf(self, corpus):
        vectorizer = TfidfVectorizer()
        matriz = vectorizer.fit_transform(corpus)
        df_tfidf = pd.DataFrame(matriz.toarray(), columns=vectorizer.get_feature_names_out(), index=[f'Texto_{i+1}' for i in range(len(corpus))])
        df_idf = pd.DataFrame({'Palabra': vectorizer.get_feature_names_out(), 'IDF': vectorizer.idf_}).sort_values('IDF', ascending=False)
        return df_tfidf, df_idf

    # 6. POS Tagging
    def etiquetar_pos(self, texto):
        doc = self.nlp(texto)
        return pd.DataFrame([{'Token': t.text, 'POS': t.pos_, 'Tag': t.tag_, 'Explicacion': spacy.explain(t.pos_)} for t in doc if not t.is_space])

    # 7. Lematización
    def lematizar(self, texto):
        doc = self.nlp(texto)
        return pd.DataFrame([{'Token': t.text, 'Lema': t.lemma_, 'POS': t.pos_} for t in doc if not t.is_space and not t.is_punct])

    # 8. Parsing Sintáctico
    def parsing_sintactico(self, texto):
        doc = self.nlp(texto)
        return pd.DataFrame([{'Token': t.text, 'Dep': t.dep_, 'Explicacion': spacy.explain(t.dep_), 'Head': t.head.text} for t in doc if not t.is_space])

    def visualizar_parsing(self, texto):
        doc = self.nlp(texto)
        displacy.render(doc, style="dep", jupyter=True, options={"distance": 100, "color": "#0284c7", "bg": "#f8fafc"})

    # 9. NER
    def reconocer_entidades(self, texto):
        doc = self.nlp(texto)
        return pd.DataFrame([{'Entidad': e.text, 'Etiqueta': e.label_, 'Explicacion': spacy.explain(e.label_)} for e in doc.ents])

    def visualizar_ner(self, texto):
        doc = self.nlp(texto)
        displacy.render(doc, style="ent", jupyter=True)

# Instanciamos el procesador en español
nlp = NLPProcesadorTexto(lang="es")
print("✓ Clase NLPProcesadorTexto esta lista para usarse.")

✓ Clase NLPProcesadorTexto esta lista para usarse.


---
## 🛠️ 1. Normalización de Texto
La **normalización** transforma el texto crudo en un formato uniforme y limpio eliminando variaciones tipográficas, signos de puntuación, mayúsculas y acentos.

In [9]:
#texto_ejemplo = "¡Hola Mundo! Este es un Ejemplo de Normalización (¡con Ácentós y Números 2026!)."
texto_limpio = nlp.normalizar_texto(texto_a_procesar)

print("🔴 Texto Original :", texto_a_procesar)
print("🟢 Texto Normalizado:", texto_limpio)

🔴 Texto Original : 
El grave incendio forestal en Villa de Leyva (Boyacá), iniciado el pasado sábado diecinueve de septiembre 
en la vereda La Castellana y el cerro San Marcos, ha consumido más de 1340 hectáreas de vegetación nativa y pino, 
afectando gravemente zonas de acuíferos, senderos turísticos y forzando la suspensión temporal de clases. 
Las autoridades, que reportan un control del fuego cercano al 78%, sospechan que la emergencia fue provocada 
por manos criminales, por lo que la Gobernación de Boyacá ha ofrecido una recompensa de hasta cuarenta y cinco millones 
de pesos por información de los responsables mientras los cuerpos de socorro y helicópteros continúan extinguiendo 
los focos activos.

🟢 Texto Normalizado: el grave incendio forestal en villa de leyva boyaca iniciado el pasado sabado diecinueve de septiembre en la vereda la castellana y el cerro san marcos ha consumido mas de 1340 hectareas de vegetacion nativa y pino afectando gravemente zonas de acuiferos senderos

---
## 🌾 2. Stemming
El **Stemming** recorta los sufijos de las palabras para obtener la raíz morfológica común (stem). Es un proceso heurístico basado en reglas de sufijo.

In [57]:
#palabras_ejemplo = ["corriendo", "correrá", "corredor", "jugadores", "jugando", "computadoras", "computación"]

stems = nlp.stemming_texto(texto_a_procesar)
df_stems = pd.DataFrame(stems, columns=["Palabra Original", "Stem (Raíz)"])
display(df_stems)


,Palabra Original,Stem (Raíz)
0,El,el
1,grave,grav
2,incendio,incendi
3,forestal,forestal
4,en,en
...,...,...
102,continúan,continu
103,extinguiendo,extingu
104,los,los
105,focos,foc


---
## 🛑 3. Stopwords (Palabras Vacías)
Las **Stopwords** son palabras altamente frecuentes (artículos, preposiciones, conjunciones) que aportan poco valor semántico en tareas de análisis.

In [32]:
#frase = ["el", "gato", "está", "durmiendo", "tranquilamente", "sobre", "el", "sofá", "de", "la", "casa"]
sin_stopwords = nlp.remover_stopwords(texto_limpio.split())

print("🔴 Texto Original :", texto_a_procesar)
print("🟢 Texto sin Stopwords:", sin_stopwords)

🔴 Texto Original : 
El grave incendio forestal en Villa de Leyva (Boyacá), iniciado el pasado sábado diecinueve de septiembre 
en la vereda La Castellana y el cerro San Marcos, ha consumido más de 1340 hectáreas de vegetación nativa y pino, 
afectando gravemente zonas de acuíferos, senderos turísticos y forzando la suspensión temporal de clases. 
Las autoridades, que reportan un control del fuego cercano al 78%, sospechan que la emergencia fue provocada 
por manos criminales, por lo que la Gobernación de Boyacá ha ofrecido una recompensa de hasta cuarenta y cinco millones 
de pesos por información de los responsables mientras los cuerpos de socorro y helicópteros continúan extinguiendo 
los focos activos.

🟢 Texto sin Stopwords: ['grave', 'incendio', 'forestal', 'villa', 'leyva', 'boyaca', 'iniciado', 'sabado', 'diecinueve', 'septiembre', 'vereda', 'castellana', 'cerro', 'san', 'marcos', 'consumido', '1340', 'hectareas', 'vegetacion', 'nativa', 'pino', 'afectando', 'gravemente', 'zonas

---
## 📊 4. Term Frequency (TF)
Mide la frecuencia con la que un término o palabra aparece en un conjunto de documentos.

In [42]:
corpus = [
    texto_a_procesar,
    #"La inteligencia artificial y el procesamiento de datos transforman el mundo.",
    #"El lenguaje natural permite la interacción humano computadora."
]

df_tf = nlp.calcular_tf(corpus)
display(df_tf)

,1340,78,activos,acuíferos,afectando,al,autoridades,boyacá,castellana,cercano,...,suspensión,sábado,temporal,turísticos,un,una,vegetación,vereda,villa,zonas
Texto_1,1,1,1,1,1,1,1,2,1,1,...,1,1,1,1,1,1,1,1,1,1


---
## 📈 5. Inverse Document Frequency (IDF / TF-IDF)
El **TF-IDF** penaliza palabras comunes a través de todos los documentos y destaca términos específicos de cada documento.

In [55]:
df_tfidf, df_idf = nlp.calcular_tfidf(corpus)

print("Matriz TF-IDF")
display(df_tfidf)

print("\n TOP 10 Inverse Document Frequency (Mayor IDF = Términos más raros/específicos)")
df_idf_order = df_idf.sort_values(by="IDF", ascending=False)
display(df_idf_order.head(10))

Matriz TF-IDF


,1340,78,activos,acuíferos,afectando,al,autoridades,boyacá,castellana,cercano,...,suspensión,sábado,temporal,turísticos,un,una,vegetación,vereda,villa,zonas
Texto_1,0.06178,0.06178,0.06178,0.06178,0.06178,0.06178,0.06178,0.12356,0.06178,0.06178,...,0.06178,0.06178,0.06178,0.06178,0.06178,0.06178,0.06178,0.06178,0.06178,0.06178



 TOP 10 Inverse Document Frequency (Mayor IDF = Términos más raros/específicos)


,Palabra,IDF
0,1340,1.0
1,78,1.0
2,activos,1.0
3,acuíferos,1.0
4,afectando,1.0
5,al,1.0
6,autoridades,1.0
7,boyacá,1.0
8,castellana,1.0
9,cercano,1.0


---
## 🏷️ 6. Part-of-Speech (POS) Tagging
El **POS Tagging** clasifica cada palabra en una categoría gramatical (Sustantivo, Verbo, Adjetivo, Adverbio, etc.).

In [60]:
#texto_pos = "Los talentosos científicos desarrollan innovadores modelos matemáticos velozmente."
df_pos = nlp.etiquetar_pos(texto_a_procesar)
display(df_pos)

,Token,POS,Tag,Explicacion
0,El,DET,DET,determiner
1,grave,ADJ,ADJ,adjective
2,incendio,NOUN,NOUN,noun
3,forestal,ADJ,ADJ,adjective
4,en,ADP,ADP,adposition
...,...,...,...,...
113,extinguiendo,VERB,VERB,verb
114,los,DET,DET,determiner
115,focos,NOUN,NOUN,noun
116,activos,ADJ,ADJ,adjective


---
## 🌿 7. Lematización
A diferencia del Stemming, la **Lematización** encuentra la forma gramatical canónica de una palabra (lema) utilizando vocabularios y análisis morfológico.

In [61]:
#texto_lem = "Los gatos estaban corriendo rápidamente y rompieron los jarrones de la mesa."
df_lem = nlp.lematizar(texto_a_procesar)
display(df_lem)

,Token,Lema,POS
0,El,el,DET
1,grave,grave,ADJ
2,incendio,incendio,NOUN
3,forestal,forestal,ADJ
4,en,en,ADP
...,...,...,...
102,continúan,continuar,VERB
103,extinguiendo,extinguir,VERB
104,los,el,DET
105,focos,foco,NOUN


---
## 🌳 8. Parsing Sintáctico (Dependency Parsing)
El **Parsing** analiza las relaciones sintácticas en la oración identificando el verbo principal (raíz), sujeto, objeto directo y modificadores dependientes.

In [66]:
texto_parse = "El grave incendio forestal en Villa de Leyva iniciado el pasado sábado diecinueve de septiembre ha consumido más de 1340 hectáreas de vegetación nativa y pino"

# 1. Tabla de dependencias
df_parse = nlp.parsing_sintactico(texto_a_procesar)
display(df_parse)

# 2. Visualización interactiva con displaCy
print("\n--- Visualización Parsing del Árbol Sintáctico ---")
nlp.visualizar_parsing(texto_parse)

,Token,Dep,Explicacion,Head
0,El,det,determiner,incendio
1,grave,amod,adjectival modifier,incendio
2,incendio,nsubj,nominal subject,consumido
3,forestal,amod,adjectival modifier,incendio
4,en,case,case marking,Villa
...,...,...,...,...
113,extinguiendo,xcomp,open clausal complement,continúan
114,los,det,determiner,focos
115,focos,obj,object,extinguiendo
116,activos,amod,adjectival modifier,focos



--- Visualización Parsing del Árbol Sintáctico ---


---
## 🔍 9. Named-Entity Recognition (NER)
**NER** reconoce e identifica entidades nombradas en el texto (Personas, Organizaciones, Lugares, Fechas, Monedas, etc.).

In [70]:
texto_ner = "El grave incendio forestal en Villa de Leyva (Boyaca) iniciado el pasado sábado diecinueve de septiembre ha consumido más de 1340 hectáreas de vegetación nativa y pino. Reporta BBC Noticias."

# 1. Tabla de Entidades
df_ner = nlp.reconocer_entidades(texto_a_procesar)
display(df_ner)

# 2. Visualización destacada con displaCy
print("\n--- Visualización de Entidades Nombradas (NER) ---")
nlp.visualizar_ner(texto_ner)

,Entidad,Etiqueta,Explicacion
0,Villa de Leyva,LOC,"Non-GPE locations, mountain ranges, bodies of ..."
1,Boyacá,LOC,"Non-GPE locations, mountain ranges, bodies of ..."
2,La Castellana,LOC,"Non-GPE locations, mountain ranges, bodies of ..."
3,cerro San Marcos,LOC,"Non-GPE locations, mountain ranges, bodies of ..."
4,Gobernación de Boyacá,LOC,"Non-GPE locations, mountain ranges, bodies of ..."



--- Visualización de Entidades Nombradas (NER) ---


---
## ⚙️ Segundo Bloque: `Large language Models`

## 🏆 9. Generación de texto usando LLM
Texto explicativo sobre concepto.

In [1]:
#CODIGO SEGUNDO BLOQUE (LLM)
print("--- Generación de texto usando LLM ---")

--- Generación de texto usando LLM ---


---
## ⚙️ Tercer Bloque: `Dataset Youtube Statistics`

## 📦 Descarga de Dataset
Ejecutar la siguiente celda para instalar y descargar el datasets de Youtube Statistics.

In [10]:
# Instalación de librerías en Google Colab
!pip install -q kagglehub

import kagglehub
import os

# Ultima version del dataset
path = kagglehub.dataset_download("advaypatil/youtube-statistics")

#Archivos disponibles en dataset: ['videos-stats.csv', 'comments.csv']
ruta_csv = os.path.join(path, "videos-stats.csv") #Seleccionar csv a trabajar
df = pd.read_csv(ruta_csv)

print("✅ Descarga de dataset exitosa.")

Using Colab cache for faster access to the 'youtube-statistics' dataset.
✅ Descarga de dataset exitosa.


## ❓ 10. Question Answering
Texto explicativo sobre concepto.

In [ ]:
#CODIGO TERCER BLOQUE (QA)